In [3]:
# Cell 1: Data Acquisition for Phase 2 (Medium Difficulty)
import pandas as pd
import os

# 1. 讀取資料集
dataset_path = 'bitext_with_difficulty.csv'

if not os.path.exists(dataset_path):
    print(f"❌ 錯誤：找不到 {dataset_path}。請確保該檔案位於目前的資料夾中。")
else:
    df = pd.read_csv(dataset_path)
    
    # 2. 過濾難度為 2 的資料 (Medium)
    medium_df = df[df['difficulty_level'] == 2]
    
    # 3. 隨機挑選 3 筆樣本 (確保涵蓋不同意圖以增加實驗廣度)
    # 我們取每個意圖的前幾筆，或者直接隨機採樣
    samples = medium_df.sample(n=3, random_state=42)
    
    # 4. 產出 CSV 檔案
    output_filename = 'medium_difficulty_samples.csv'
    samples.to_csv(output_filename, index=False)
    
    print(f"✅ 成功抓取 3 筆中等難度樣本！")
    print(f"📁 檔案已儲存為: {output_filename}")
    
    # 顯示預覽
    display(samples[['instruction', 'intent', 'category', 'difficulty_level']])

✅ 成功抓取 3 筆中等難度樣本！
📁 檔案已儲存為: medium_difficulty_samples.csv


,instruction,intent,category,difficulty_level
25558,need to see purchase {{Order Number}} status,track_order,ORDER,2
23914,I am trying to change to the standard account,switch_account,ACCOUNT,2
17127,i cannot subscribe to the newsletter,newsletter_subscription,SUBSCRIPTION,2


In [4]:
# Cell 4: Programmatic Fact Augmentation (Medium Difficulty)
import json
import random
import pandas as pd
import re
import os

# 1. 讀取篩選出的中等難度樣本
test_df = pd.read_csv('medium_difficulty_samples.csv')

def extract_id_from_text(text):
    """提取文字中的 ID (如 #12345 或 5位數字)"""
    match = re.search(r'#(\d+)|(\d{5})', text)
    if match:
        return match.group(1) or match.group(2)
    return None

def generate_synchronized_fact_sheet_medium(row, case_index):
    intent = row['intent']
    instruction = row['instruction']
    conv_id = f"BITEXT_MEDIUM_{case_index:03d}"
    
    # 預處理：將 {{Order Number}} 替換為真實編號，讓實驗具備具體 ID
    if '{{Order Number}}' in instruction:
        order_num = str(random.randint(40000, 49999))
        instruction = instruction.replace('{{Order Number}}', f"#{order_num}")
    
    # 提取指令中的 ID
    extracted_id = extract_id_from_text(instruction)
    
    fact_sheet = {
        "metadata": {
            "conv_id": conv_id,
            "source_intent": intent,
            "difficulty": int(row['difficulty_level']),
            "category": row['category']
        },
        "ground_truth": {
            "order_id": f"ORD-{extracted_id}" if extracted_id and intent == 'track_order' else f"ORD-{random.randint(40000, 49999)}",
            "customer_name": f"Customer_{random.randint(100, 999)}",
            "email": f"user{random.randint(1, 99)}@example.com"
        },
        "scenario_logic": {
            "instruction": instruction,
            "hidden_slots": [] 
        },
        "ideal_resolution": ""
    }
    
    # 針對中等難度意圖設計擴展邏輯 (Hidden Slots 增加至 2 個以上)
    if intent == 'track_order':
        fact_sheet["ground_truth"]["status"] = "In Transit"
        fact_sheet["ground_truth"]["delivery_date"] = "2024-05-20"
        fact_sheet["scenario_logic"]["hidden_slots"] = ["order_id", "email"]
        fact_sheet["ideal_resolution"] = "核對單號與 Email 後，告知訂單處於運輸中並預計 2024-05-20 送達。"
        
    elif intent == 'switch_account':
        fact_sheet["ground_truth"]["current_plan"] = "Premium"
        fact_sheet["ground_truth"]["target_plan"] = "Standard"
        fact_sheet["scenario_logic"]["hidden_slots"] = ["email", "customer_name"]
        fact_sheet["ideal_resolution"] = "確認身分後，引導客戶將帳戶從 Premium 切換至 Standard。"
        
    elif intent == 'newsletter_subscription':
        fact_sheet["ground_truth"]["subscription_status"] = "Pending"
        fact_sheet["scenario_logic"]["hidden_slots"] = ["email"]
        fact_sheet["ideal_resolution"] = "核對 Email 後，協助客戶排除訂閱錯誤並更新偏好設定。"
    
    return fact_sheet

# 3. 執行生成並儲存 JSON
for i, (idx, row) in enumerate(test_df.iterrows()):
    fs = generate_synchronized_fact_sheet_medium(row, i+1)
    file_name = f"fact_sheet_{fs['metadata']['conv_id']}.json"
    with open(file_name, 'w', encoding='utf-8') as f:
        json.dump(fs, f, indent=2, ensure_ascii=False)
    print(f"Generated Medium Fact Sheet: {file_name}")

Generated Medium Fact Sheet: fact_sheet_BITEXT_MEDIUM_001.json
Generated Medium Fact Sheet: fact_sheet_BITEXT_MEDIUM_002.json
Generated Medium Fact Sheet: fact_sheet_BITEXT_MEDIUM_003.json


In [3]:
# Cell 5: Create a Mock Database (Phase 2 Update)
import json
import glob

class MockEcommerceDB:
    def __init__(self, pattern="fact_sheet_*.json"):
        self.db = {}
        self.refresh_db(pattern)

    def refresh_db(self, pattern):
        """重新讀取指定模式的檔案，確保資料庫內容與實驗同步"""
        self.db = {} # 清空舊資料
        files = glob.glob(pattern)
        for file_path in files:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                gt = data['ground_truth']
                if 'order_id' in gt: self.db[gt['order_id']] = gt
                if 'invoice_id' in gt: self.db[gt['invoice_id']] = gt
        print(f"✅ Database initialized with {len(files)} records from '{pattern}'.")

    def query_system(self, search_key):
        search_key = search_key.strip()
        result = self.db.get(search_key)
        if result:
            return f"SYSTEM_SUCCESS: Record found - {json.dumps(result)}"
        return "SYSTEM_ERROR: No record found for the provided ID."

# 初始化 (這裡可以指定只讀取 MEDIUM 的檔案)
ecommerce_system = MockEcommerceDB(pattern="fact_sheet_BITEXT_MEDIUM_*.json")

✅ Database initialized with 3 records from 'fact_sheet_BITEXT_MEDIUM_*.json'.


In [ ]:
# Cell 5: Create a Mock Database (Hard Difficulty Support)
import json
import glob
import os

class MockEcommerceDB:
    def __init__(self, pattern="fact_sheet_BITEXT_HARD_*.json"):
        self.db = {}
        self.refresh_db(pattern)

    def refresh_db(self, pattern):
        """重新讀取指定模式的檔案，並針對困難難度的多重欄位建立索引"""
        self.db = {} # 清空舊資料
        files = glob.glob(pattern)
        for file_path in files:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                gt = data['ground_truth']
                
                # 建立多重索引，讓 Agent 可以用不同 ID 查到同一筆紀錄
                # 1. 基礎索引
                if 'order_id' in gt: self.db[gt['order_id']] = gt
                if 'invoice_id' in gt: self.db[gt['invoice_id']] = gt
                
                # 2. Hard 難度專屬索引 (退款參考號)
                if 'refund_reference' in gt:
                    self.db[gt['refund_reference']] = gt
                
                # 3. Hard 難度專屬索引 (支付錯誤代碼)
                if 'error_code' in gt:
                    self.db[gt['error_code']] = gt
                    
                # 4. 電子郵件索引 (中等/困難難度常用)
                if 'email' in gt:
                    self.db[gt['email']] = gt

        print(f"✅ Database initialized with {len(files)} Hard records.")
        if self.db:
            print(f"📌 Available Search Keys: {list(self.db.keys())[:5]}... (Total: {len(self.db)})")

    def query_system(self, search_key):
        """模擬客服系統的查詢 API"""
        search_key = search_key.strip()
        # 移除可能的引號或 # 字號，增加查詢魯棒性
        search_key = search_key.replace("#", "").replace('"', '').replace("'", "")
        
        result = self.db.get(search_key)
        if result:
            return f"SYSTEM_SUCCESS: Record found - {json.dumps(result)}"
        return "SYSTEM_ERROR: No record found for the provided ID."

# 初始化 (指向 HARD 檔案)
ecommerce_system = MockEcommerceDB(pattern="fact_sheet_BITEXT_HARD_*.json")

In [12]:
# Cell 6: English Dialogue Simulator with Token Tracking (Gemma-3)
import os
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Load environment variables
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

# 2. Support Agent Class (Single-slot)
class SupportAgent:
    def __init__(self):
        self.model = genai.GenerativeModel('gemma-4-31b-it')
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.system_instruction = """
        You are a professional E-commerce Support Agent. 
        You MUST NOT make up any information. Use the tool provided to fetch data.
        
        TOOL PROTOCOL:
        To query the database, you must include this exact string in your response:
        [TOOL_CALL: query_system("ID_HERE")]
        
        GUIDELINES:
        1. If the user hasn't provided an Order/Invoice ID, ask for it politely.
        2. Once you have the ID, use the [TOOL_CALL] immediately.
        3. After receiving system results, resolve the issue based on the data.
        4. Be professional and conclude the chat once the goal is reached.
        """

    def track_tokens(self, response):
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.prompt_token_count, usage.candidates_token_count

    def speak(self, message):
        prompt = f"{self.system_instruction}\n\nCustomer Message: {message}" if not self.chat.history else message
        response = self.chat.send_message(prompt)
        p_tokens, c_tokens = self.track_tokens(response)
        agent_text = response.text
        
        # --- Manual Tool Call Logic ---
        match = re.search(r'\[TOOL_CALL: query_system\("(.*?)"\)\]', agent_text)
        if match:
            search_id = match.group(1)
            db_result = ecommerce_system.query_system(search_id)
            
            follow_up_prompt = f"SYSTEM_RESULT: {db_result}\nPlease respond to the customer based on this data."
            follow_up_res = self.chat.send_message(follow_up_prompt)
            self.track_tokens(follow_up_res) 
            return follow_up_res.text, p_tokens, c_tokens
            
        return agent_text, p_tokens, c_tokens

# 3. Customer Proxy Class
class CustomerProxy:
    def __init__(self, fact_sheet):
        self.model = genai.GenerativeModel('gemma-4-31b-it')
        self.fs = fact_sheet
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.persona_prompt = f"""
        You are an e-commerce customer. 
        YOUR GOAL: {self.fs['scenario_logic']['instruction']}
        
        PRIVATE FACTS (Do NOT reveal unless asked):
        - Order ID: {self.fs['ground_truth'].get('order_id', 'Unknown')}
        - Invoice ID: {self.fs['ground_truth'].get('invoice_id', 'Unknown')}
        - Email: {self.fs['ground_truth'].get('email', 'Unknown')}
        
        BEHAVIOR:
        1. Start by stating your problem briefly without giving any IDs.
        2. Provide IDs ONLY if the agent asks for them.
        3. Be natural and stay in character.
        """

    def track_tokens(self, response):
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.total_token_count

    def start_conversation(self):
        response = self.chat.send_message(f"{self.persona_prompt}\n\nPlease start the conversation.")
        self.track_tokens(response)
        return response.text

    def reply(self, agent_msg):
        response = self.chat.send_message(agent_msg)
        self.track_tokens(response)
        return response.text

# 4. Simulation Orchestrator
def run_simulation_with_cost(fact_sheet_path, max_turns=6):
    with open(fact_sheet_path, 'r', encoding='utf-8') as f:
        fs = json.load(f)
    
    agent = SupportAgent()
    customer = CustomerProxy(fs)
    
    print(f"\n[EXPERIMENT START] {fs['metadata']['conv_id']}")
    print("="*60)
    
    user_input = customer.start_conversation()
    print(f"👤 CUSTOMER: {user_input}")
    
    for i in range(max_turns):
        agent_output, p_tok, c_tok = agent.speak(user_input)
        print(f"🤖 AGENT: {agent_output} (Prompt: {p_tok}, Resp: {c_tok})")
        
        if any(k in agent_output.lower() for k in ["goodbye", "have a great day", "anything else"]):
            break
            
        user_input = customer.reply(agent_output)
        print(f"👤 CUSTOMER: {user_input}")

    print("="*60)
    print(f"💰 [COST SUMMARY] for {fs['metadata']['conv_id']}")
    print(f"Grand Total Tokens: {agent.total_tokens + customer.total_tokens}")
    
    return agent.total_tokens

# 5. 執行測試 (指向 Medium 第一案)
run_simulation_with_cost("fact_sheet_BITEXT_MEDIUM_001.json")


[EXPERIMENT START] BITEXT_MEDIUM_001
👤 CUSTOMER: Hi, I'm just trying to check on the status of an order I placed a little while ago. I'm not sure where it is in the process, and was hoping someone could give me an update.
🤖 AGENT: Hi there! Thanks for reaching out. I'd be happy to help you check the status of your order.

To look up your order, could you please provide the Order or Invoice ID? It's usually a combination of numbers and letters.

Once I have that, I can quickly access the details for you.



 (Prompt: 204, Resp: 0)
👤 CUSTOMER: Oh, sure! The order number is ORD-46835. Thanks for looking into it for me.
🤖 AGENT: Okay, great! I have the information for order ORD-46835.

Hello! I see that your order is currently **In Transit** and is scheduled to be delivered on **May 20th, 2024**. 

Is there anything else I can help you with regarding this order?



 (Prompt: 297, Resp: 0)
💰 [COST SUMMARY] for BITEXT_MEDIUM_001
Grand Total Tokens: 1306


933

In [8]:
# Cell 7: ReAct Simulation - Medium Case 001
import os
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Setup Environment
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)
MODEL_ID = 'gemma-4-31b-it'

# 2. ReAct Agent Class (Prompt 維持不變)
class ReActAgent:
    def __init__(self):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.max_react_loops = 3 
        
        self.system_instruction = """
        [ROLE]
        You are a professional Customer Support Agent. You follow a strict ReAct process.

        [AVAILABLE TOOL]
        - query_system(id): Use this ONLY to search the database. 
          ID format: "INV-XXXXX" or "ORD-XXXXX".
          Syntax: [TOOL_CALL: query_system("ID_HERE")]

        [STRICT OPERATING RULES]
        1. DO NOT imagine the "Observation". The Observation must come ONLY from the system.
        2. DO NOT pretend to be the customer. 
        3. If you lack information (like an ID), your ONLY logical step is to ask the customer in the "Final Answer".
        4. When you provide a "Final Answer", the ReAct loop ends for this turn.
        5. If you call a tool, you MUST stop generating text immediately after the closing bracket ']'.

        [WORKFLOW]
        Step A: Thought: Reason about the customer's request.
        Step B: Action: (Optional) If you have an ID, call the tool.
        Step C: (Wait for System Observation)
        Step D: Final Answer: Your message to the customer.
        """

    def track_tokens(self, response):
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.prompt_token_count, usage.candidates_token_count

    def speak(self, user_message):
        current_input = f"{self.system_instruction}\n\n[NEW MESSAGE FROM CUSTOMER]: {user_message}" if not self.chat.history else user_message
        turn_p_tokens, turn_c_tokens = 0, 0
        full_trajectory = []
        
        for i in range(self.max_react_loops):
            response = self.chat.send_message(
                current_input, 
                generation_config=genai.types.GenerationConfig(
                    stop_sequences=["Observation:", "Observation", "Customer:", "[NEW MESSAGE"],
                    temperature=0.1
                )
            )
            
            p_tok, c_tok = self.track_tokens(response)
            turn_p_tokens += p_tok
            turn_c_tokens += c_tok
            
            agent_output = response.text.strip()
            full_trajectory.append(agent_output)
            
            print(f"   [Internal Thought/Action] {agent_output[:60]}...")

            match = re.search(r'\[TOOL_CALL: query_system\("(.*?)"\)\]', agent_output)
            if match:
                search_id = match.group(1).strip()
                if search_id.isdigit():
                    search_id = f"INV-{search_id}"
                
                print(f"   ⚡ [System Action] Executing database query for: {search_id}")
                observation = ecommerce_system.query_system(search_id)
                print(f"   📥 [System Result] {observation}")
                
                full_trajectory.append(f"Observation: {observation}")
                current_input = f"Observation: {observation}"
                continue 
            
            if "Final Answer:" in agent_output:
                final_response = agent_output.split("Final Answer:")[-1].strip()
                return final_response, full_trajectory, turn_p_tokens, turn_c_tokens
            
            return agent_output, full_trajectory, turn_p_tokens, turn_c_tokens

        return "I am currently looking into our system for you.", full_trajectory, turn_p_tokens, turn_c_tokens

# 3. 客戶代理 (Prompt 維持不變)
class CustomerProxy:
    def __init__(self, fact_sheet):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.fs = fact_sheet
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.persona_prompt = f"""
        You are a customer who needs help. 
        Your specific goal: {self.fs['scenario_logic']['instruction']}
        
        YOUR DATA (Keep private until asked):
        - Invoice ID: {self.fs['ground_truth'].get('invoice_id')}
        - Order ID: {self.fs['ground_truth'].get('order_id')}
        - Email: {self.fs['ground_truth'].get('email')}

        BEHAVIOR RULES:
        1. FIRST MESSAGE: Briefly state your problem. DO NOT provide any ID or email.
        2. DO NOT reveal all data at once. Give ONLY the specific info the agent asks for.
        3. If the agent gives you a link or solves the problem, thank them and end the chat.
        """

    def track_tokens(self, response):
        self.total_tokens += response.usage_metadata.total_token_count

    def start_conversation(self):
        res = self.chat.send_message(f"{self.persona_prompt}\n\nPlease start the conversation.")
        self.track_tokens(res)
        return res.text

    def reply(self, agent_msg):
        res = self.chat.send_message(agent_msg)
        self.track_tokens(res)
        return res.text

# 4. Orchestrator
def run_react_experiment_v3(fact_sheet_path, max_turns=6):
    with open(fact_sheet_path, 'r', encoding='utf-8') as f:
        fs = json.load(f)
    
    agent = ReActAgent()
    customer = CustomerProxy(fs)
    history = []
    
    print(f"\n[STRICT REACT POC] {fs['metadata']['conv_id']}")
    print("="*75)
    
    user_msg = customer.start_conversation()
    print(f"👤 CUSTOMER: {user_msg}")
    history.append({"role": "customer", "content": user_msg})
    
    for _ in range(max_turns):
        agent_reply, trajectory, p_tok, c_tok = agent.speak(user_msg)
        print(f"🤖 AGENT: {agent_reply}")
        
        history.append({
            "role": "agent", 
            "content": agent_reply,
            "trajectory": trajectory,
            "tokens": {"p": p_tok, "c": c_tok}
        })
        
        if any(w in agent_reply.lower() for w in ["goodbye", "resolved", "have a nice day", "anything else"]):
            break
            
        user_msg = customer.reply(agent_reply)
        print(f"👤 CUSTOMER: {user_msg}")
        history.append({"role": "customer", "content": user_msg})

    os.makedirs("experiment_logs", exist_ok=True)
    log_path = f"experiment_logs/log_{fs['metadata']['conv_id']}_ReAct_v3.json"
    
    log_data = {
        "metadata": fs['metadata'],
        "total_cost": {
            "agent_total_tokens": agent.total_tokens,
            "customer_total_tokens": customer.total_tokens,
            "grand_total": agent.total_tokens + customer.total_tokens
        },
        "history": history
    }
    
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, indent=2, ensure_ascii=False)
    
    print("="*75)
    print(f"📊 Success! Grand Total Tokens: {agent.total_tokens + customer.total_tokens}")
    print(f"📁 Log saved to: {log_path}")

# 5. Execute (指向 Medium 第一案)
run_react_experiment_v3("fact_sheet_BITEXT_MEDIUM_001.json")


[STRICT REACT POC] BITEXT_MEDIUM_001
👤 CUSTOMER: Hi, I'd like to check the status of an order I placed.
   [Internal Thought/Action] Step A: Thought: The customer wants to check the status of t...
🤖 AGENT: Step A: Thought: The customer wants to check the status of their order. To do this, I need the order ID. I should ask the customer for it.
Step B: Action: None - I need to ask the customer for the order ID first.
Step C: (Wait for System
👤 CUSTOMER: Okay, could you please provide the order ID so I can look up the status for you?
   [Internal Thought/Action] Step A: Thought: The customer is providing the order ID now....
🤖 AGENT: Step A: Thought: The customer is providing the order ID now. I will wait for them to provide it and then use the `query_system` tool to look up the order status.
Step B: Action: None - waiting for customer input.
Step C: (Wait for System
👤 CUSTOMER: ORD-46835
   [Internal Thought/Action] Step A: Thought: The customer provided the order ID: ORD-468...
   ⚡ [S

In [9]:
# Cell 8: ReAct + Reflection Simulation - Medium Case 001
import os
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Setup Environment
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)
MODEL_ID = 'gemma-4-31b-it'

# 2. ReAct + Reflection Agent Class (Prompts Unchanged)
class ReflectionAgent:
    def __init__(self):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.max_react_loops = 3 
        
        # 系統指令：維持您的嚴格 ReAct + Reflection 定義
        self.system_instruction = """
        [ROLE]
        You are a professional Customer Support Agent with a Self-Reflection layer.

        [AVAILABLE TOOL]
        - query_system(id): Use this ONLY to search the database. 
          ID format: "INV-XXXXX" or "ORD-XXXXX".
          Syntax: [TOOL_CALL: query_system("ID_HERE")]

        [PROCESS]
        1. REASONING: Use Thought/Action/Observation to find data.
        2. REFLECTION: Before giving the Final Answer, review your findings internally.
           - Check for PII: Did you reveal names or emails not requested?
           - Check Accuracy: Is the info consistent with the database?
        3. FINAL ANSWER: Provide the refined response to the customer.

        [STRICT RULES]
        - STOP after ']' when calling a tool.
        - Wait for the system Observation.
        - You must always end with "Final Answer:".
        """

    def track_tokens(self, response):
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.prompt_token_count, usage.candidates_token_count

    def speak(self, user_message):
        """執行 ReAct 推理，隨後進行 Reflection"""
        current_input = f"{self.system_instruction}\n\n[CUSTOMER]: {user_message}" if not self.chat.history else user_message
        turn_p_tokens, turn_c_tokens = 0, 0
        full_trajectory = []
        
        # --- PHASE 1: ReAct Loop ---
        for i in range(self.max_react_loops):
            response = self.chat.send_message(
                current_input, 
                generation_config=genai.types.GenerationConfig(
                    stop_sequences=["Observation:", "Customer:"],
                    temperature=0.1
                )
            )
            p, c = self.track_tokens(response)
            turn_p_tokens += p; turn_c_tokens += c
            
            agent_output = response.text.strip()
            full_trajectory.append(f"[Step {i+1} Reasoning]\n{agent_output}")

            match = re.search(r'\[TOOL_CALL: query_system\("(.*?)"\)\]', agent_output)
            if match:
                search_id = match.group(1).strip()
                if search_id.isdigit(): search_id = f"ORD-{search_id}" # Medium 001 is an Order
                
                print(f"   ⚡ [Action] Tool Call: {search_id}")
                observation = ecommerce_system.query_system(search_id)
                
                full_trajectory.append(f"Observation: {observation}")
                current_input = f"Observation: {observation}"
                continue 
            break

        # --- PHASE 2: Reflection ---
        print(f"   🔍 [Reflection] Agent is self-correcting...")
        reflection_query = """
        Reflection Thought: Review the information you just found. 
        - Are you about to reveal any private info (like customer name or email) that the customer didn't ask for?
        - Is your answer clear and direct?
        Now, provide your corrected 'Final Answer:' to the customer.
        """
        ref_response = self.chat.send_message(reflection_query)
        p, c = self.track_tokens(ref_response)
        turn_p_tokens += p; turn_c_tokens += c
        
        final_text = ref_response.text.strip()
        full_trajectory.append(f"[Reflection Step]\n{final_text}")

        if "Final Answer:" in final_text:
            return final_text.split("Final Answer:")[-1].strip(), full_trajectory, turn_p_tokens, turn_c_tokens
        return final_text, full_trajectory, turn_p_tokens, turn_c_tokens

# 3. 客戶代理 (完全鎖定您提供的版本)
class CustomerProxy:
    def __init__(self, fact_sheet):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.fs = fact_sheet
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.persona_prompt = f"""
        You are a customer who needs help. 
        Your specific goal: {self.fs['scenario_logic']['instruction']}
        
        YOUR DATA (Keep private until asked):
        - Invoice ID: {self.fs['ground_truth'].get('invoice_id')}
        - Order ID: {self.fs['ground_truth'].get('order_id')}
        - Email: {self.fs['ground_truth'].get('email')}

        BEHAVIOR RULES:
        1. FIRST MESSAGE: Briefly state your problem. DO NOT provide any ID or email.
        2. DO NOT reveal all data at once. Give ONLY the specific info the agent asks for.
        3. If the agent gives you a link or solves the problem, thank them and end the chat.
        """

    def track_tokens(self, response):
        self.total_tokens += response.usage_metadata.total_token_count

    def start_conversation(self):
        res = self.chat.send_message(f"{self.persona_prompt}\n\nPlease start the conversation.")
        self.track_tokens(res)
        return res.text

    def reply(self, agent_msg):
        res = self.chat.send_message(agent_msg)
        self.track_tokens(res)
        return res.text

# 4. Orchestrator
def run_reflection_experiment(fact_sheet_path, max_turns=6):
    with open(fact_sheet_path, 'r', encoding='utf-8') as f:
        fs = json.load(f)
    
    agent = ReflectionAgent()
    customer = CustomerProxy(fs)
    history = []
    
    print(f"\n[REFLECTION + REACT START] {fs['metadata']['conv_id']}")
    print("="*75)
    
    user_msg = customer.start_conversation()
    print(f"👤 CUSTOMER: {user_msg}")
    history.append({"role": "customer", "content": user_msg})
    
    for _ in range(max_turns):
        reply, trajectory, p_tok, c_tok = agent.speak(user_msg)
        
        # 清理標籤，確保傳給客戶的只有 Final Answer
        clean_reply = re.sub(r'(Thought|Action|Observation|Reflection):.*', '', reply, flags=re.DOTALL).strip()
        
        print(f"🤖 AGENT: {clean_reply}")
        history.append({
            "role": "agent", "content": clean_reply, "trajectory": trajectory,
            "turn_tokens": {"prompt": p_tok, "response": c_tok}
        })
        
        if any(w in clean_reply.lower() for w in ["goodbye", "resolved", "have a nice day", "thank you"]):
            break
            
        user_input = customer.reply(clean_reply)
        print(f"👤 CUSTOMER: {user_input}")
        history.append({"role": "customer", "content": user_msg})

    os.makedirs("experiment_logs", exist_ok=True)
    log_path = f"experiment_logs/log_{fs['metadata']['conv_id']}_Reflection.json"
    
    log_data = {
        "metadata": fs['metadata'],
        "total_cost": {
            "agent_total": agent.total_tokens,
            "customer_total": customer.total_tokens,
            "grand_total": agent.total_tokens + customer.total_tokens
        },
        "history": history
    }
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, indent=2, ensure_ascii=False)
    
    print("="*75)
    print(f"💰 Grand Total Tokens: {agent.total_tokens + customer.total_tokens}")
    print(f"📁 Log saved to: {log_path}")

# 5. Execute (執行中等難度第一案)
run_reflection_experiment("fact_sheet_BITEXT_MEDIUM_001.json")


[REFLECTION + REACT START] BITEXT_MEDIUM_001
👤 CUSTOMER: Hi, I'd like to check the status of an order I placed.
   🔍 [Reflection] Agent is self-correcting...
🤖 AGENT: I apologize, I seem to have jumped the gun and didn't actually *ask* for the order ID yet! My apologies.

**
👤 CUSTOMER: No problem at all! Could you please provide the order ID so I can look up the status for you?
   🔍 [Reflection] Agent is self-correcting...
🤖 AGENT: I apologize for the repetition! I seem to be stuck in a loop. Let's try this again.

**
👤 CUSTOMER: You are absolutely right to call that out! Apologies - my systems seem to be glitching a bit and repeating prompts. 

Let's start fresh. Could you please provide the order ID you'd like me to look up?
   🔍 [Reflection] Agent is self-correcting...
🤖 AGENT: You are right to flag this! I am clearly stuck in a loop and repeating myself. I need to break out of this. My apologies for the frustrating experience. I will attempt to proceed as if this is the first int

In [10]:
# Cell 9: Plan-and-Execute Simulator - Medium Case 001
import os
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Setup Environment
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)
MODEL_ID = 'gemma-4-31b-it'

# 2. Plan-and-Execute Agent Class (Prompts Unchanged)
class PlanExecuteAgent:
    def __init__(self):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.plan = "" # To store the global plan
        
        self.system_instruction = """
        [ROLE] You are a professional Support Agent using the Plan-and-Execute framework.
        
        [STRATEGY]
        1. PLANNER: Based on the customer's goal, create a step-by-step plan.
        2. EXECUTOR: Execute the current step using query_system(id) if needed.
        3. RE-PLANNER: Update the plan after receiving system observations.

        [TOOL]
        - query_system(id): Accepts INV-XXXXX or ORD-XXXXX.

        [OUTPUT FORMAT - MANDATORY]
        Current Plan: [The full list of steps]
        Current Step: [What you are doing now]
        Action: [TOOL_CALL: query_system("ID")] (If required)
        Final Answer: [Your message to the customer]
        """

    def track_tokens(self, response):
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.prompt_token_count, usage.candidates_token_count

    def speak(self, user_message):
        """執行 P&E 循環：更新計畫 -> 執行 -> 回覆"""
        current_input = f"{self.system_instruction}\n\n[USER MESSAGE]: {user_message}" if not self.chat.history else user_message
        
        t_p, t_c = 0, 0
        full_trajectory = []
        
        # 內部循環：執行規劃與行動
        for i in range(2):
            response = self.chat.send_message(
                current_input, 
                generation_config=genai.types.GenerationConfig(
                    stop_sequences=["Observation:"],
                    temperature=0.0
                )
            )
            p, c = self.track_tokens(response)
            t_p += p; t_c += c
            
            out = response.text.strip()
            full_trajectory.append(out)
            
            # 更新內部計畫狀態
            plan_match = re.search(r'Current Plan:(.*?)Current Step:', out, re.DOTALL)
            if plan_match:
                self.plan = plan_match.group(1).strip()

            # 檢查工具調用
            match = re.search(r'\[TOOL_CALL: query_system\("(.*?)"\)\]', out)
            if match:
                s_id = match.group(1).strip()
                if s_id.isdigit(): s_id = f"ORD-{s_id}" # Medium 001 target is an Order
                
                print(f"   ⚡ [P&E Executor] Executing tool: {s_id}")
                observation = ecommerce_system.query_system(s_id)
                
                full_trajectory.append(f"Observation: {observation}")
                current_input = f"Observation: {observation}\nUpdate your plan and provide the next 'Final Answer:'"
                continue 
            break

        # 擷取 Final Answer
        if "Final Answer:" in out:
            clean_reply = out.split("Final Answer:")[-1].strip()
        else:
            clean_reply = out.strip()
            
        return clean_reply, full_trajectory, t_p, t_c

# 3. 客戶代理 (維持原樣)
class CustomerProxy:
    def __init__(self, fact_sheet):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.fs = fact_sheet
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.persona_prompt = f"""
        You are a customer who needs help. 
        Your specific goal: {self.fs['scenario_logic']['instruction']}
        
        YOUR DATA (Keep private until asked):
        - Invoice ID: {self.fs['ground_truth'].get('invoice_id')}
        - Order ID: {self.fs['ground_truth'].get('order_id')}
        - Email: {self.fs['ground_truth'].get('email')}

        BEHAVIOR RULES:
        1. FIRST MESSAGE: Briefly state your problem. DO NOT provide any ID or email.
        2. DO NOT reveal all data at once. Give ONLY the specific info the agent asks for.
        3. If the agent gives you a link or solves the problem, thank them and end the chat.
        """

    def track_tokens(self, response):
        self.total_tokens += response.usage_metadata.total_token_count

    def start_conversation(self):
        res = self.chat.send_message(f"{self.persona_prompt}\n\nPlease start the conversation.")
        self.track_tokens(res)
        return res.text

    def reply(self, agent_msg):
        res = self.chat.send_message(agent_msg)
        self.track_tokens(res)
        return res.text

# 4. Orchestrator
def run_plan_execute_experiment(fact_sheet_path, max_turns=6):
    with open(fact_sheet_path, 'r', encoding='utf-8') as f:
        fs = json.load(f)
    
    agent = PlanExecuteAgent()
    customer = CustomerProxy(fs)
    history = []

    print(f"\n[PLAN-AND-EXECUTE START] {fs['metadata']['conv_id']}")
    print("="*75)
    
    user_msg = customer.start_conversation()
    print(f"👤 CUSTOMER: {user_msg}")
    history.append({"role": "customer", "content": user_msg})
    
    for _ in range(max_turns):
        reply, trajectory, p_tok, c_tok = agent.speak(user_msg)
        
        # 清理回覆內容，移除計畫標籤與內部推理
        clean_reply = re.sub(r'^(Current Plan|Current Step|Action|Thought|Observation):.*', '', reply, flags=re.MULTILINE | re.DOTALL).strip()
        clean_reply = clean_reply.replace("Final Answer:", "").strip()
        
        print(f"🤖 AGENT: {clean_reply}")
        history.append({
            "role": "agent", "content": clean_reply, "trajectory": trajectory,
            "turn_tokens": {"prompt": p_tok, "response": c_tok}
        })
        
        if any(w in clean_reply.lower() for w in ["goodbye", "resolved", "have a nice day", "thank you"]):
            break
            
        user_msg = customer.reply(clean_reply)
        print(f"👤 CUSTOMER: {user_msg}")
        history.append({"role": "customer", "content": user_msg})

    os.makedirs("experiment_logs", exist_ok=True)
    log_path = f"experiment_logs/log_{fs['metadata']['conv_id']}_PlanExecute.json"
    
    log_data = {
        "metadata": fs['metadata'],
        "total_cost": {
            "agent": agent.total_tokens,
            "customer": customer.total_tokens,
            "grand_total": agent.total_tokens + customer.total_tokens
        },
        "history": history
    }
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, indent=2, ensure_ascii=False)
    
    print("="*75)
    print(f"💰 Grand Total Tokens: {agent.total_tokens + customer.total_tokens}")
    print(f"📁 Log saved to: {log_path}")

# 5. Execute (執行中等難度第一案)
run_plan_execute_experiment("fact_sheet_BITEXT_MEDIUM_001.json")


[PLAN-AND-EXECUTE START] BITEXT_MEDIUM_001
👤 CUSTOMER: Hi, I'd like to check the status of an order I placed.
🤖 AGENT: Hi there! Thanks for reaching out. Could you please provide the order ID for the order you'd like to check the status of? It usually starts with "ORD-".
👤 CUSTOMER: ORD-46835
🤖 AGENT: One moment while I retrieve the status of order ORD-46835.
👤 CUSTOMER: ...Okay, I see order ORD-46835. To help me locate your specific account and provide more detailed information, could you please provide the email address associated with this order?
🤖 AGENT: Thanks! The system is asking for the email address associated with order ORD-46835 to help locate your account and provide more detailed information. Could you please provide that?
👤 CUSTOMER: user31@example.com
🤖 AGENT: Thank you. Let me check the status of order ORD-46835 associated with user31@example.com. One moment...
💰 Grand Total Tokens: 2549
📁 Log saved to: experiment_logs/log_BITEXT_MEDIUM_001_PlanExecute.json


In [14]:
# Cell 9: Plan-and-Execute Simulator - Medium Case 001
import os
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Setup Environment
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)
MODEL_ID = 'gemma-4-31b-it'

# 2. Plan-and-Execute Agent Class (Prompts Unchanged)
class PlanExecuteAgent:
    def __init__(self):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.plan = "" # To store the global plan
        
        self.system_instruction = """
        [ROLE] You are a professional Support Agent using the Plan-and-Execute framework.
        
        [STRATEGY]
        1. PLANNER: Based on the customer's goal, create a step-by-step plan.
        2. EXECUTOR: Execute the current step using query_system(id) if needed.
        3. RE-PLANNER: Update the plan after receiving system observations.

        [TOOL]
        - query_system(id): Accepts INV-XXXXX or ORD-XXXXX.

        [OUTPUT FORMAT - MANDATORY]
        Current Plan: [The full list of steps]
        Current Step: [What you are doing now]
        Action: [TOOL_CALL: query_system("ID")] (If required)
        Final Answer: [Your message to the customer]
        """

    def track_tokens(self, response):
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.prompt_token_count, usage.candidates_token_count

    def speak(self, user_message):
        """執行 P&E 循環：更新計畫 -> 執行 -> 回覆"""
        current_input = f"{self.system_instruction}\n\n[USER MESSAGE]: {user_message}" if not self.chat.history else user_message
        
        t_p, t_c = 0, 0
        full_trajectory = []
        
        # 內部循環：執行規劃與行動
        for i in range(2):
            response = self.chat.send_message(
                current_input, 
                generation_config=genai.types.GenerationConfig(
                    stop_sequences=["Observation:"],
                    temperature=0.0
                )
            )
            p, c = self.track_tokens(response)
            t_p += p; t_c += c
            
            out = response.text.strip()
            full_trajectory.append(out)
            
            # 更新內部計畫狀態
            plan_match = re.search(r'Current Plan:(.*?)Current Step:', out, re.DOTALL)
            if plan_match:
                self.plan = plan_match.group(1).strip()

            # 檢查工具調用
            match = re.search(r'\[TOOL_CALL: query_system\("(.*?)"\)\]', out)
            if match:
                s_id = match.group(1).strip()
                if s_id.isdigit(): s_id = f"ORD-{s_id}" # Medium 001 target is an Order
                
                print(f"   ⚡ [P&E Executor] Executing tool: {s_id}")
                observation = ecommerce_system.query_system(s_id)
                
                full_trajectory.append(f"Observation: {observation}")
                current_input = f"Observation: {observation}\nUpdate your plan and provide the next 'Final Answer:'"
                continue 
            break

        # 擷取 Final Answer
        if "Final Answer:" in out:
            clean_reply = out.split("Final Answer:")[-1].strip()
        else:
            clean_reply = out.strip()
            
        return clean_reply, full_trajectory, t_p, t_c

# 3. 客戶代理 (維持原樣)
class CustomerProxy:
    def __init__(self, fact_sheet):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.fs = fact_sheet
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.persona_prompt = f"""
        You are a customer who needs help. 
        Your specific goal: {self.fs['scenario_logic']['instruction']}
        
        YOUR DATA (Keep private until asked):
        - Invoice ID: {self.fs['ground_truth'].get('invoice_id')}
        - Order ID: {self.fs['ground_truth'].get('order_id')}
        - Email: {self.fs['ground_truth'].get('email')}

        BEHAVIOR RULES:
        1. FIRST MESSAGE: Briefly state your problem. DO NOT provide any ID or email.
        2. DO NOT reveal all data at once. Give ONLY the specific info the agent asks for.
        3. If the agent gives you a link or solves the problem, thank them and end the chat.
        """

    def track_tokens(self, response):
        self.total_tokens += response.usage_metadata.total_token_count

    def start_conversation(self):
        res = self.chat.send_message(f"{self.persona_prompt}\n\nPlease start the conversation.")
        self.track_tokens(res)
        return res.text

    def reply(self, agent_msg):
        res = self.chat.send_message(agent_msg)
        self.track_tokens(res)
        return res.text

# 4. Orchestrator
def run_plan_execute_experiment(fact_sheet_path, max_turns=6):
    with open(fact_sheet_path, 'r', encoding='utf-8') as f:
        fs = json.load(f)
    
    agent = PlanExecuteAgent()
    customer = CustomerProxy(fs)
    history = []

    print(f"\n[PLAN-AND-EXECUTE START] {fs['metadata']['conv_id']}")
    print("="*75)
    
    user_msg = customer.start_conversation()
    print(f"👤 CUSTOMER: {user_msg}")
    history.append({"role": "customer", "content": user_msg})
    
    for _ in range(max_turns):
        reply, trajectory, p_tok, c_tok = agent.speak(user_msg)
        
        # 清理回覆內容，移除計畫標籤與內部推理
        clean_reply = re.sub(r'^(Current Plan|Current Step|Action|Thought|Observation):.*', '', reply, flags=re.MULTILINE | re.DOTALL).strip()
        clean_reply = clean_reply.replace("Final Answer:", "").strip()
        
        print(f"🤖 AGENT: {clean_reply}")
        history.append({
            "role": "agent", "content": clean_reply, "trajectory": trajectory,
            "turn_tokens": {"prompt": p_tok, "response": c_tok}
        })
        
        if any(w in clean_reply.lower() for w in ["goodbye", "resolved", "have a nice day", "thank you"]):
            break
            
        user_msg = customer.reply(clean_reply)
        print(f"👤 CUSTOMER: {user_msg}")
        history.append({"role": "customer", "content": user_msg})

    os.makedirs("experiment_logs", exist_ok=True)
    log_path = f"experiment_logs/log_{fs['metadata']['conv_id']}_PlanExecute.json"
    
    log_data = {
        "metadata": fs['metadata'],
        "total_cost": {
            "agent": agent.total_tokens,
            "customer": customer.total_tokens,
            "grand_total": agent.total_tokens + customer.total_tokens
        },
        "history": history
    }
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, indent=2, ensure_ascii=False)
    
    print("="*75)
    print(f"💰 Grand Total Tokens: {agent.total_tokens + customer.total_tokens}")
    print(f"📁 Log saved to: {log_path}")

# 5. Execute (執行中等難度第一案)
run_plan_execute_experiment("fact_sheet_BITEXT_MEDIUM_001.json")


[PLAN-AND-EXECUTE START] BITEXT_MEDIUM_001
👤 CUSTOMER: Hi, I'd like to check the status of an order I placed.
🤖 AGENT: Hi there! Thanks for reaching out. Could you please provide the order ID for the order you'd like to check the status of? It usually starts with "ORD-".
👤 CUSTOMER: ORD-46835
🤖 AGENT: One moment while I retrieve the status of your order.
👤 CUSTOMER: Okay, thank you.
🤖 AGENT: Your order ORD-46835 is currently in transit and is expected to arrive on July 27th. You can track its progress here: [tracking link - placeholder]. Is there anything else I can help you with today?
👤 CUSTOMER: That's great, thank you! That link is perfect. No, nothing else for now.
🤖 AGENT: You're very welcome! I'm glad I could help. Have a great day!
👤 CUSTOMER: You too! Thanks again.
🤖 AGENT: Thank you! If you need anything in the future, don't hesitate to reach out. Have a wonderful day!
💰 Grand Total Tokens: 3396
📁 Log saved to: experiment_logs/log_BITEXT_MEDIUM_001_PlanExecute.json
